# FarmSLM

Small decoder language model for farming. The context window is **500,000 tokens**. Weights are written as **safetensors**.

Documents are conditioned with one domain token:

- `english`
- `agricultural_linguistics`
- `chemistry`
- `weather_reading`
- `physics`

## How 500,000 tokens fit

The model does not build a 500,000 by 500,000 attention matrix. It reads the document in segments of **2,048** tokens. Each of the **8** layers carries **256** memory slots from one segment to the next. After a segment, those slots are rewritten by cross-attention over that segment. The next segment can see them. Rotary positions are the absolute index in the full document, so position 499,999 is valid.

Inside a segment, attention is causal. Memory slots from earlier segments are visible. Future tokens are not. The carried state is `8 x 256 x 512` values per batch item, about 2 MB in fp16, instead of a key/value cache for all 500,000 tokens.

## Layer stack

Input ids have shape `[batch, time]`.

1. Token embedding, vocabulary 512, hidden size 512. The output head uses the same matrix. A map chip is split into 16-pixel patches and placed in front of these tokens, so the words are predicted with the sheet in view.
2. For each segment, for each layer:
   - RMSNorm, then grouped-query attention. 8 query heads, 2 key/value heads, head size 64. Rotary theta is 1,000,000.
   - Residual add.
   - RMSNorm, then SwiGLU with intermediate width 1,536.
   - Residual add.
   - Cross-attention updates that layer's 256 memory slots for the next segment.
3. Final RMSNorm and the tied output head produce next-token logits `[batch, time, 512]`.

Token ids 0–255 are UTF-8 bytes. 256 is pad, 257 is start, 258 is end, and 259–263 are the five domain tokens.

The stack is 30,687,744 parameters. The numbers live in `farm_slm.kdl`. The modules live in `farm_slm/model.py`.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "farm_slm").is_dir():
    raise SystemExit("Run this notebook from the Field directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from farm_slm.config import load_project
from farm_slm.model import FarmSLM, architecture_report

project = load_project()
model = FarmSLM(project.model)
print(architecture_report(model))
print(model)


## Data collection

The GitHub queries are empty until you add them. `collect_github` searches repositories, keeps Markdown, reStructuredText, plain text, and notebook sources, and writes JSONL under `data/corpus/`.

Each record stores `domain`, `text`, `repo`, `path`, `sha`, `license`, and `source_url`. The collector does not run repository code. It skips private-key files, `.env` files, and documents that contain a private key or a GitHub token. Set `GITHUB_TOKEN` in the environment when you are ready to collect; the token is not written into the corpus.

```python
plan["chemistry"] = ["soil cation exchange capacity"]
plan["weather_reading"] = ["agricultural weather forecast"]
summary = collect_github(plan)
```

In [ ]:
from farm_slm.collect import collection_section, empty_plan

plan = empty_plan()
print(collection_section(plan))


## Weights

Training uses the local NVIDIA, AMD, or Apple Metal accelerator when one is present, and the CPU otherwise. It reads `data/corpus/*.jsonl` when those files exist. Until the collection slots have queries, it trains on a short primer so this cell can still write `artifacts/farm_slm.safetensors`. Raise `train.steps` in `farm_slm.kdl` after the corpus is in place. Load only a safetensors file this notebook wrote.

In [ ]:
from safetensors import safe_open

from farm_slm.train import run_training

weights = run_training()
with safe_open(weights, framework="pt", device="cpu") as handle:
    print(handle.metadata())
